In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score


In [5]:
df=pd.read_csv("Dataset\covid.csv")
df.head()

,id,sex,patient_type,entry_date,date_symptoms,date_died,intubed,pneumonia,age,pregnancy,...,inmsupr,hypertension,other_disease,cardiovascular,obesity,renal_chronic,tobacco,contact_other_covid,covid_res,icu
0,16169f,2,1,04-05-2020,02-05-2020,9999-99-99,97,2,27,97,...,2,2,2,2,2,2,2,2,1,97
1,1009bf,2,1,19-03-2020,17-03-2020,9999-99-99,97,2,24,97,...,2,2,2,2,2,2,2,99,1,97
2,167386,1,2,06-04-2020,01-04-2020,9999-99-99,2,2,54,2,...,2,2,2,2,1,2,2,99,1,2
3,0b5948,2,2,17-04-2020,10-04-2020,9999-99-99,2,1,30,97,...,2,2,2,2,2,2,2,99,1,2
4,0d01b5,1,2,13-04-2020,13-04-2020,22-04-2020,2,2,60,2,...,2,1,2,1,2,2,2,99,1,2


In [6]:
df.columns

Index(['id', 'sex', 'patient_type', 'entry_date', 'date_symptoms', 'date_died',
       'intubed', 'pneumonia', 'age', 'pregnancy', 'diabetes', 'copd',
       'asthma', 'inmsupr', 'hypertension', 'other_disease', 'cardiovascular',
       'obesity', 'renal_chronic', 'tobacco', 'contact_other_covid',
       'covid_res', 'icu'],
      dtype='object')

In [7]:
df=df.drop(['id','entry_date','date_symptoms'],axis=1)
df.head()

,sex,patient_type,date_died,intubed,pneumonia,age,pregnancy,diabetes,copd,asthma,inmsupr,hypertension,other_disease,cardiovascular,obesity,renal_chronic,tobacco,contact_other_covid,covid_res,icu
0,2,1,9999-99-99,97,2,27,97,2,2,2,2,2,2,2,2,2,2,2,1,97
1,2,1,9999-99-99,97,2,24,97,2,2,2,2,2,2,2,2,2,2,99,1,97
2,1,2,9999-99-99,2,2,54,2,2,2,2,2,2,2,2,1,2,2,99,1,2
3,2,2,9999-99-99,2,1,30,97,2,2,2,2,2,2,2,2,2,2,99,1,2
4,1,2,22-04-2020,2,2,60,2,1,2,2,2,1,2,1,2,2,2,99,1,2


In [8]:
df=df.replace(2,0)

In [9]:
# Create death column from date_died
df['death'] = df['date_died'].apply(lambda x: 0 if pd.isna(x) else 1)

# Create severity
df['severity'] = ((df['icu'] == 1) | 
                  (df['intubed'] == 1) | 
                  (df['death'] == 1)).astype(int)

df['severity'].value_counts()


severity
1    566602
Name: count, dtype: int64

In [10]:
X=df.drop(['severity','date_died','death'],axis=1)
y=df['severity']
X.head()

,sex,patient_type,intubed,pneumonia,age,pregnancy,diabetes,copd,asthma,inmsupr,hypertension,other_disease,cardiovascular,obesity,renal_chronic,tobacco,contact_other_covid,covid_res,icu
0,0,1,97,0,27,97,0,0,0,0,0,0,0,0,0,0,0,1,97
1,0,1,97,0,24,97,0,0,0,0,0,0,0,0,0,0,99,1,97
2,1,0,0,0,54,0,0,0,0,0,0,0,0,1,0,0,99,1,0
3,0,0,0,1,30,97,0,0,0,0,0,0,0,0,0,0,99,1,0
4,1,0,0,0,60,0,1,0,0,0,1,0,1,0,0,0,99,1,0


In [11]:
from sklearn.feature_selection import SelectKBest, chi2

selector = SelectKBest(score_func=chi2, k=10)
selector.fit(X, y)

chi_scores = pd.DataFrame({
    "Feature": X.columns,
    "Score": selector.scores_
}).sort_values(by="Score", ascending=False)

chi_scores


,Feature,Score
0,sex,NaN
1,patient_type,NaN
2,intubed,NaN
3,pneumonia,NaN
4,age,NaN
5,pregnancy,NaN
6,diabetes,NaN
7,copd,NaN
8,asthma,NaN
9,inmsupr,NaN


In [12]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X, y)

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

importance


,Feature,Importance
0,sex,0.0
1,patient_type,0.0
2,intubed,0.0
3,pneumonia,0.0
4,age,0.0
5,pregnancy,0.0
6,diabetes,0.0
7,copd,0.0
8,asthma,0.0
9,inmsupr,0.0


In [13]:
df['severity'].value_counts()

severity
1    566602
Name: count, dtype: int64

In [14]:
df['date_died'].unique()[:5]

array(['9999-99-99', '22-04-2020', '29-04-2020', '21-05-2020',
       '28-04-2020'], dtype=object)

In [15]:
df['death'] = df['date_died'].apply(lambda x: 0 if x == '9999-99-99' else 1)

df['death'].value_counts()



death
0    530426
1     36176
Name: count, dtype: int64

In [16]:
df['severity'] = ((df['icu'] == 1) | 
                  (df['intubed'] == 1) | 
                  (df['death'] == 1)).astype(int)

df['severity'].value_counts()


severity
0    522230
1     44372
Name: count, dtype: int64

In [17]:
df[['icu', 'intubed']].value_counts().head()


icu  intubed
97   97         444689
0    0          106721
1    0            5102
     1            5010
0    1            4955
Name: count, dtype: int64

In [18]:
df['icu'] = df['icu'].apply(lambda x: 1 if x == 1 else 0)
df['intubed'] = df['intubed'].apply(lambda x: 1 if x == 1 else 0)


In [19]:
df['severity'] = ((df['icu'] == 1) | 
                  (df['intubed'] == 1) | 
                  (df['death'] == 1)).astype(int)

df['severity'].value_counts()


severity
0    522230
1     44372
Name: count, dtype: int64

In [20]:
from sklearn.feature_selection import SelectKBest, chi2

selector = SelectKBest(score_func=chi2, k=10)
selector.fit(X, y)

chi_scores = pd.DataFrame({
    "Feature": X.columns,
    "Score": selector.scores_
}).sort_values(by="Score", ascending=False)

chi_scores


,Feature,Score
0,sex,NaN
1,patient_type,NaN
2,intubed,NaN
3,pneumonia,NaN
4,age,NaN
5,pregnancy,NaN
6,diabetes,NaN
7,copd,NaN
8,asthma,NaN
9,inmsupr,NaN


In [21]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X, y)

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

importance


,Feature,Importance
0,sex,0.0
1,patient_type,0.0
2,intubed,0.0
3,pneumonia,0.0
4,age,0.0
5,pregnancy,0.0
6,diabetes,0.0
7,copd,0.0
8,asthma,0.0
9,inmsupr,0.0


In [22]:
# Recreate X and y AFTER fixing severity
X = df.drop(['severity', 'date_died', 'death'], axis=1)
y = df['severity']

# Check
print(X.shape)
print(y.value_counts())


(566602, 19)
severity
0    522230
1     44372
Name: count, dtype: int64


In [23]:
for col in X.columns:
    print(col, X[col].nunique())


sex 2
patient_type 2
intubed 2
pneumonia 3
age 119
pregnancy 4
diabetes 3
copd 3
asthma 3
inmsupr 3
hypertension 3
other_disease 3
cardiovascular 3
obesity 3
renal_chronic 3
tobacco 3
contact_other_covid 3
covid_res 3
icu 2


In [24]:
X = X.loc[:, X.nunique() > 1]


In [25]:
X.dtypes


sex                    int64
patient_type           int64
intubed                int64
pneumonia              int64
age                    int64
pregnancy              int64
diabetes               int64
copd                   int64
asthma                 int64
inmsupr                int64
hypertension           int64
other_disease          int64
cardiovascular         int64
obesity                int64
renal_chronic          int64
tobacco                int64
contact_other_covid    int64
covid_res              int64
icu                    int64
dtype: object

In [26]:
X = X.apply(pd.to_numeric)


In [27]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X, y)

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

importance.head(10)


,Feature,Importance
4,age,0.230781
18,icu,0.163786
1,patient_type,0.152852
2,intubed,0.144567
3,pneumonia,0.122558
17,covid_res,0.041772
16,contact_other_covid,0.034900
6,diabetes,0.016114
10,hypertension,0.014256
13,obesity,0.013943


In [28]:
X = df.drop(['severity', 'date_died', 'death', 'icu', 'intubed'], axis=1)
y = df['severity']


In [29]:
rf = RandomForestClassifier()
rf.fit(X, y)

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

importance


,Feature,Importance
3,age,0.345584
1,patient_type,0.222638
2,pneumonia,0.170694
16,covid_res,0.052761
15,contact_other_covid,0.037098
5,diabetes,0.024991
9,hypertension,0.023568
12,obesity,0.019930
14,tobacco,0.017171
10,other_disease,0.015519


In [30]:
selected_features = [
    'age',
    'patient_type',
    'pneumonia',
    'covid_res',
    'diabetes',
    'hypertension',
    'obesity',
    'cardiovascular',
    'renal_chronic'
]


In [31]:
X_selected = df[selected_features]
y = df['severity']

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))


Accuracy: 0.9241711598026844


In [32]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.94      0.98      0.96    104370
           1       0.53      0.33      0.41      8951

    accuracy                           0.92    113321
   macro avg       0.74      0.65      0.68    113321
weighted avg       0.91      0.92      0.92    113321



In [33]:
model = RandomForestClassifier(class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.99      0.86      0.92    104370
           1       0.35      0.87      0.50      8951

    accuracy                           0.86    113321
   macro avg       0.67      0.86      0.71    113321
weighted avg       0.94      0.86      0.89    113321



In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [35]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_gb))
print(classification_report(y_test, y_pred_gb))


Accuracy: 0.9288834373152373
              precision    recall  f1-score   support

           0       0.95      0.98      0.96    104447
           1       0.58      0.33      0.42      8874

    accuracy                           0.93    113321
   macro avg       0.76      0.66      0.69    113321
weighted avg       0.92      0.93      0.92    113321



In [36]:
pip install xgboost


     ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
     ---------------------------------------- 0.0/101.7 MB 2.0 MB/s eta 0:00:50
     ---------------------------------------- 0.1/101.7 MB 1.3 MB/s eta 0:01:17
     ---------------------------------------- 0.2/101.7 MB 1.8 MB/s eta 0:00:57
     ---------------------------------------- 0.8/101.7 MB 4.5 MB/s eta 0:00:23
     ---------------------------------------- 1.2/101.7 MB 5.6 MB/s eta 0:00:18
      --------------------------------------- 1.3/101.7 MB 4.9 MB/s eta 0:00:21
      --------------------------------------- 1.5/101.7 MB 4.9 MB/s eta 0:00:21
      --------------------------------------- 1.6/101.7 MB 4.5 MB/s eta 0:00:23
      --------------------------------------- 1.7/101.7 MB 4.2 MB/s eta 0:00:24
      --------------------------------------- 1.8/101.7 MB 4.1 MB/s eta 0:00:25
      --------------------------------------- 1.9/101.7 MB 3.7 MB/s eta 0:00:27
      --------------------------------------- 2


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight= (522230 / 44372),  # imbalance ratio
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))


Accuracy: 0.856090221582937
              precision    recall  f1-score   support

           0       0.99      0.85      0.92    104447
           1       0.35      0.95      0.51      8874

    accuracy                           0.86    113321
   macro avg       0.67      0.90      0.71    113321
weighted avg       0.94      0.86      0.88    113321



In [38]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight={0:1, 1:2},  # mild balancing, not extreme
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.9160349802772655
              precision    recall  f1-score   support

           0       0.97      0.94      0.95    104447
           1       0.47      0.60      0.53      8874

    accuracy                           0.92    113321
   macro avg       0.72      0.77      0.74    113321
weighted avg       0.93      0.92      0.92    113321

